# Model

- Split to quadrants.

- Split each quadrant by office.

- Detect Positions location.

---------NEW--------
- Find top row for each column.

    Define as the same height of position/ Month character (月,年, etc).
    
- Crawl down top row of each column.

- Split to wages and names by length of character sequence.

In [30]:
origin='C:/Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/'

import sys
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')

In [31]:
#Set up Environment
import os,cv2
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Extract_Data')
from Extract import DetectCols, ToDict, AssignIndex
from Split import HoriPart, VertPart
from Read import Vision
from google.cloud import vision
from google.cloud.vision_v1 import types

sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
from OCR import Clova
api_url='https://1srlcnzf08.apigw.ntruss.com/custom/v1/2412/9a859f9b3a7d952aad17f388d1a445041f8aef0f1eccc6fcce8d3a856272fcbd/general'
secret_key='eEhyV0NGRlRLVXpZVWZnWFNDamRVaWFYZ1NSQ1NKSFI='


os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = origin+'Tokyo_Jobs\\Environment\\GoogleVision\\practice-302516-01e0f7245b03.json'
# Instantiates a client
client = vision.ImageAnnotatorClient()

In [32]:
import json, os, cv2
import pandas as pd

Year,Showa='1942','17'
Quality='Line'

df = pd.read_csv(origin+'/Tokyo_Jobs/Processed_Data/Index/S'+Showa+'.csv')
df=df.drop(df.columns[0], axis=1)



# Step A. Set up Page

In [34]:
Page=35
file_path2='Tokyo_Jobs/Raw_Data/Metropolitan_DA/1942/'+str(Quality)+'/Page'+'{:03d}'.format(Page)+'.jpg'
img=cv2.imread(os.path.join(origin,file_path2))

cv2.namedWindow("Resized_Window", cv2.WINDOW_NORMAL)
cv2.resizeWindow("Resized_Window", 1280, 1440)
cv2.imshow("Resized_Window",img)
cv2.waitKey(0)
cv2.destroyAllWindows()


#Convert to book page
BookPage=2*Page-14
Rows=df[(df['Page']==BookPage)]
NxRows=df[(df['Page']==BookPage+1)]
if len(Rows)==0:
    print('No Office at Right Side. Page'+str(BookPage))
if len(NxRows)==0:
    print('No Office at Left Side. Page'+str(BookPage+1))
        
texts=Vision(img,'zh',client)
textsCLOVA=Clova(origin,Page,api_url,secret_key)

# Step B. Get Office Info

In [36]:
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Page')
from Split import VertPart
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\General')
from PageCut import HoriPart
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Office')
from Organize import Filter, ConvertDict
path2=os.path.join(origin,file_path2)

HH,WW=img.shape[:2]
HoriPoint=HoriPart(img,Page,texts)[0]
VertPoint=1*HH//3

##Right Page
BoxR=Filter(BookPage,texts,HoriPoint)
BoxR=ConvertDict(BoxR)

##Left Page
BoxL=Filter(BookPage+1,texts,HoriPoint)
BoxL=ConvertDict(BoxL)

In [37]:
Dict={}
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Detect')
from Organize import FilterBox
from Detect import DetectOffice
LocList=[]

#RightPage
OfficeList=Rows['Office'].tolist()
for Office in OfficeList:
    LocList.append(DetectOffice(BoxR, Office,VertPoint))
    BoxR=FilterBox(BoxR,LocList,VertPoint)

#LeftPage
OfficeList=NxRows['Office'].tolist()
for Office in OfficeList:
    LocList.append(DetectOffice(BoxL, Office,VertPoint))
    BoxL=FilterBox(BoxL,LocList,VertPoint)

Dict[Page]=LocList

In [7]:
LocList

[{'CommonWord': ['久', '病', '院', '大', '保'],
  'WordCount': 5,
  'Words': '大久保病院從橋區東',
  'Box': {'description': '〇',
   'bounding_poly': vertices {
     x: 863
     y: 188
   }
   vertices {
     x: 879
     y: 188
   }
   vertices {
     x: 879
     y: 208
   }
   vertices {
     x: 863
     y: 208
   },
   'Index': 12},
  'OfficeName': '大久保病院'},
 {'CommonWord': ['塚', '院', '大', '病'],
  'WordCount': 4,
  'Words': '大塚病院',
  'Box': {'description': '大',
   'bounding_poly': vertices {
     x: 358
     y: 215
   }
   vertices {
     x: 372
     y: 215
   }
   vertices {
     x: 372
     y: 225
   }
   vertices {
     x: 358
     y: 225
   },
   'Index': 42},
  'OfficeName': '大塚病院'}]

# Step C. Get Position Info

In [38]:
import itertools
#Split quardrant into offices and detect Positions
sys.path.append(origin+'Tokyo_Jobs\\Data_Collection\\Split_Position')
from OrganizePosi import Split, SplitClova, Crop, CropClova
from DetectPosi import DetectPosi,DetectPosiClova,AggregatePosi

CrossWalk=pd.read_csv(origin+"Tokyo_Jobs/Processed_Data/PositionCrosswalk.csv")['Japanese']
Positions=CrossWalk.tolist()
PosiDict['Pre'+str(Page)]=[]
PosiDict_CLOVA['Pre'+str(Page)]=[]

DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)
DF_CLOVA=CropClova(textsCLOVA,HoriPoint,VertPoint,Dict,Page)

##UR
BoxList,BoxListCLOVA=Split(DF['UR']['Box'],DF['UR']['Thres']),SplitClova(DF_CLOVA['UR']['Box'],DF_CLOVA['UR']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UR']['Thres'],PosiDict_CLOVA,Positions)

##LR
BoxList,BoxListCLOVA=Split(DF['LR']['Box'],DF['LR']['Thres']),SplitClova(DF_CLOVA['LR']['Box'],DF_CLOVA['LR']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LR']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LR']['Thres'],PosiDict_CLOVA,Positions)

##UL
BoxList,BoxListCLOVA=Split(DF['UL']['Box'][1:],DF['UL']['Thres']),SplitClova(DF_CLOVA['UL']['Box'],DF_CLOVA['UL']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['UL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['UL']['Thres'],PosiDict_CLOVA,Positions)

#LL
BoxList,BoxListCLOVA=Split(DF['LL']['Box'],DF['LL']['Thres']),SplitClova(DF_CLOVA['LL']['Box'],DF_CLOVA['LL']['Thres'])
PosiDict,PosiDict_CLOVA=DetectPosi(BoxList,DF['LL']['Thres'],PosiDict,Positions),DetectPosiClova(BoxListCLOVA,DF_CLOVA['LL']['Thres'],PosiDict_CLOVA,Positions)

PosiDict=AggregatePosi(PosiDict,PosiDict_CLOVA)

PosiDict={Office: list(itertools.chain(*PosiDict[Office])) for Office in PosiDict}

ImportError: cannot import name 'SplitClova' from 'OrganizePosi' (C:\Users/Keitaro Ninomiya/Box/Research Notes (keitaro2@illinois.edu)/Tokyo_Jobs\Data_Collection\Split_Position\OrganizePosi.py)

In [ ]:
PosiDict

# Main Code

- Specify Quadrant.

- Extract position level texts.

    sub-dict from position dictionary.

- Assign texts to positions $\times$ offices.

- Organize texts to columns.

- Test Improvement by Japanese OCR.

In [8]:
DF=Crop(texts,HoriPoint,VertPoint,Dict,Page)

In [9]:
from OrganizeName import FilterPosi, RenewPosiList,GetOffice
from Extract import ExtractName

In [ ]:
NameDict={}
#UR
try:    
    BoxList=Split(DF['UR']['Box'],DF['UR']['Thres'])[0]
    OfficeList=GetOffice('UR','Pre',PosiDict,HoriPoint,VertPoint)
    NameDict['UR']=ExtractName(BoxList,OfficeList,PosiDict,HoriPoint,VertPoint,NameDict,'UR','NA',img)
except:
    print('UR'+'Failed')
#LR
try:    
    BoxList=Split(DF['LR']['Box'],DF['LR']['Thres'])[0]
    OfficeList=GetOffice('LR','UR',PosiDict,HoriPoint,VertPoint)
    NameDict['LR']=ExtractName(BoxList,OfficeList,PosiDict,HoriPoint,VertPoint,NameDict,'LR','UR',img)
except:
    print('LR'+'Failed')

#UL
try:
    BoxList=Split(DF['UL']['Box'][1:],DF['UL']['Thres'])[0]
    OfficeList=GetOffice('UL','LR',PosiDict,HoriPoint,VertPoint)
    NameDict['UL']=ExtractName(BoxList,OfficeList,PosiDict,HoriPoint,VertPoint,NameDict,'UL','LR',img)
except:
    print('UL'+'Failed')

#LL
try:    
    BoxList=Split(DF['LL']['Box'],DF['LL']['Thres'])[0]
    OfficeList=GetOffice('LL','UL',PosiDict,HoriPoint,VertPoint)
    NameDict['LL']=ExtractName(BoxList,OfficeList,PosiDict,HoriPoint,VertPoint,NameDict,'LL','UL',img)
except:
    print('LL'+'Failed')


In [ ]:
from Extract import CountSize
NoList=['UR','LR','UL','LL']

OfficeLists=[d for d in list(NameDict.keys()) if d not in NoList]
list(map(CountSize,OfficeLists,[NameDict]*len(OfficeLists)))
